# Part 2. Core 기초

---

## 4장. Engine과 Connection

### 4.1 Engine과 Connection의 관계

3장에서 Engine을 만들었다. 이제 Engine이 실제로 어떻게 동작하는지 들여다볼 차례다. Engine은 직접 SQL을 실행하지 않는다. Engine은 **연결 팩토리**일 뿐이며, 실제 SQL 실행은 **Connection** 객체가 담당한다.

```
Engine ──── 연결 풀 관리, 방언 처리
   │
   ├──→ Connection 1 ──── SQL 실행, 트랜잭션
   ├──→ Connection 2 ──── SQL 실행, 트랜잭션
   └──→ Connection 3 ──── SQL 실행, 트랜잭션
```

Connection은 데이터베이스와의 단일 통신 채널을 나타낸다. SQLAlchemy의 Connection은 내부적으로 DBAPI 드라이버의 connection 객체를 감싸고 있으며, 트랜잭션 상태와 컨텍스트를 추적한다.

### 4.2 Connection 얻기: `engine.connect()`

Engine에서 Connection을 얻는 가장 기본적인 방법은 `connect()`다.

In [1]:
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///:memory:", echo=True)

with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print(result.scalar()) # 쿼리 실행 결과에서 첫 번째 행(row)의 첫 번째 컬럼 값 하나 리턴, 행이 없으면 None 리턴

2026-05-29 15:04:23,578 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-29 15:04:23,578 INFO sqlalchemy.engine.Engine SELECT 1
2026-05-29 15:04:23,579 INFO sqlalchemy.engine.Engine [generated in 0.00089s] ()
1
2026-05-29 15:04:23,579 INFO sqlalchemy.engine.Engine ROLLBACK


반드시 `with` 문(컨텍스트 매니저)으로 감싸야 한다. 컨텍스트가 종료되면 Connection이 자동으로 연결 풀로 반환된다. 명시적으로 `conn.close()`를 호출하는 것과 같은 효과지만, 예외가 발생해도 안전하다!.  (※ PyMySql 에선 Connection 이 with 구문이 끝나도 close 되지 않았기에 명시적으로 close() 롤 호출해야만 했다!)

여기서 매우 중요한 사실 하나!. **`engine.connect()`로 얻은 Connection에서 `execute()`만 호출하고 끝나면 변경 사항이 커밋되지 않는다.** 1.x 버전의 "autocommit" 동작은 2.0에서 완전히 제거되었다. 명시적으로 커밋해야 한다.

### 4.3 트랜잭션 관리: 두 가지 스타일

SQLAlchemy 2.0은 트랜잭션을 다루는 두 가지 패턴을 제공한다.

**1) Commit as you go (사용하면서 커밋)**

작업 중간중간 자유롭게 `commit()`과 `rollback()`을 호출하는 방식이다. 여러 트랜잭션을 하나의 Connection에서 처리할 수 있다.

```python
with engine.connect() as conn:
    conn.execute(text("INSERT INTO users (name) VALUES ('Alice')"))
    conn.commit()  # 첫 번째 트랜잭션 커밋
    
    # 새로운 트랜잭션이 암묵적으로 시작됨
    conn.execute(text("INSERT INTO users (name) VALUES ('Bob')"))
    conn.rollback()  # 두 번째 트랜잭션 롤백
    
    # 또 다른 새 트랜잭션
    conn.execute(text("INSERT INTO users (name) VALUES ('Charlie')"))
    conn.commit()
```

`commit()` 또는 `rollback()` 직후, 다음 `execute()`가 호출되면 새 트랜잭션이 자동으로 시작된다. 이를 **autobegin**이라 한다.

**2) Begin once (한 번 시작)**

전체 블록을 하나의 트랜잭션으로 묶는 방식이다. 블록이 정상 종료되면 자동 커밋, 예외가 발생하면 자동 롤백된다.

```python
with engine.begin() as conn:
    conn.execute(text("INSERT INTO users (name) VALUES ('Alice')"))
    conn.execute(text("INSERT INTO users (name) VALUES ('Bob')"))
    # 블록 종료 시 자동 COMMIT
    # 예외 발생 시 자동 ROLLBACK
```

`engine.connect()`가 아니라 `engine.begin()`을 사용한다는 점에 주목하자.


### 4.4 어떤 스타일을 써야 하는가

공식 문서는 **begin once 스타일을 권장**한다. 코드가 짧고, 트랜잭션 경계가 명확하며, 예외 처리가 자동으로 이루어지기 때문이다. 대부분의 경우 이 스타일이면 충분하다.

**commit as you go**는 다음과 같은 경우에 유용하다.

- 여러 독립적인 트랜잭션을 한 연결로 처리할 때
- 트랜잭션 중간에 사용자 입력을 기다리는 등 동적인 흐름이 필요할 때
- 학습이나 디버깅 목적으로 각 단계의 커밋 시점을 명확히 확인하고 싶을 때

이 튜토리얼에서는 양쪽을 모두 보여주되, 실전 예제는 begin once를 기본으로 한다.

### 4.5 `Result` 객체 다루기

`conn.execute()`는 `Result` 객체를 반환한다. 이 객체에서 다양한 방법으로 데이터를 꺼낼 수 있다.

```python
with engine.connect() as conn:
    result = conn.execute(text("SELECT id, name FROM users"))
    
    # 1) 모든 행을 리스트로
    rows = result.all()
    
    # 2) 한 행씩 순회
    for row in result:
        print(row.id, row.name)
    
    # 3) 정확히 한 행 (없거나 둘 이상이면 예외)
    row = result.one()
    
    # 4) 한 행 또는 None
    row = result.one_or_none()
    
    # 5) 첫 행만 (나머지는 버림)
    row = result.first()
    
    # 6) 단일 스칼라 값 (첫 행의 첫 컬럼)
    value = result.scalar()
```

`Result`는 한 번만 순회할 수 있다. 위 예시들을 한 번에 모두 호출하려고 하면 두 번째부터는 빈 결과가 나오니 주의하자.


### 4.6 `Row` 객체

결과의 각 행은 `Row` 객체다. `Row`는 named tuple과 비슷하게 동작한다.

```python
with engine.connect() as conn:
    result = conn.execute(text("SELECT id, name FROM users"))
    for row in result:
        # 인덱스 접근
        print(row[0], row[1])
        
        # 속성 접근
        print(row.id, row.name)
        
        # 언패킹
        id_, name = row
        
        # 딕셔너리로 변환
        print(row._mapping["id"])
```

딕셔너리로 다루고 싶다면 `mappings()`를 사용한다.

```python
with engine.connect() as conn:
    result = conn.execute(text("SELECT id, name FROM users"))
    for row_dict in result.mappings():
        print(row_dict["id"], row_dict["name"])
```


---

## 5장. 텍스트 기반 SQL 실행


### 5.1 `text()` 구문이 필요한 이유

지금까지 우리는 `conn.execute(text("..."))` 형태로 SQL을 실행했다. 왜 문자열을 그냥 넘기지 않고 `text()`로 감싸는 걸까?

SQLAlchemy는 원칙적으로 **임의의 문자열을 SQL로 실행하지 않는다.** 이는 의도된 안전장치다. `text()`는 "이것은 명시적으로 작성된 SQL 텍스트입니다"라고 선언하는 역할을 한다. 이 명시성 덕분에 실수로 사용자 입력을 SQL로 실행하는 사고를 막을 수 있다.

```python
from sqlalchemy import text

stmt = text("SELECT * FROM users WHERE active = 1")
```

`text()`로 만든 객체는 단순한 문자열이 아니라 SQL 표현식 객체다. 컴파일, 바인딩, 최적화 등이 적용된다.


### 5.2 파라미터 바인딩

SQL 인젝션을 막기 위해, 외부 입력은 반드시 **파라미터 바인딩**으로 전달해야 한다. 절대 문자열 포매팅이나 f-string으로 SQL을 만들지 말자.

**💥잘못된 예 (SQL 인젝션 취약):**

```python
# 절대 이렇게 하지 말 것
user_input = "Alice' OR '1'='1"
stmt = text(f"SELECT * FROM users WHERE name = '{user_input}'")
```

**✅올바른 예:**

```python
stmt = text("SELECT * FROM users WHERE name = :name")
with engine.connect() as conn:
    result = conn.execute(stmt, {"name": "Alice"})
```

`:name`이 바인드 파라미터다. 두 번째 인자로 딕셔너리를 넘기면 SQLAlchemy가 안전하게 값을 바인딩한다.


### 5.3 여러 행 한 번에 실행하기

INSERT 같은 작업은 한 번에 여러 행을 처리하는 경우가 많다. 파라미터 딕셔너리의 리스트를 넘기면 된다.

```python
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO users (name, age) VALUES (:name, :age)"),
        [
            {"name": "Alice", "age": 30},
            {"name": "Bob", "age": 25},
            {"name": "Charlie", "age": 35},
        ],
    )
```

이를 **executemany** 스타일이라 부른다. SQLAlchemy는 가능한 경우 내부적으로 최적화하여 단일 INSERT 문으로 묶어 실행한다(PostgreSQL, SQLite, MariaDB, Oracle 등에서). 이는 2.0의 `insertmanyvalues` 기능 덕분이며, 1.x 대비 INSERT 성능이 크게 개선된 이유다.


### 5.4 SELECT 결과 활용

In [4]:
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///:memory:", echo=True)

with engine.begin() as conn:
    # 테이블 생성
    result = conn.execute(text("""
        CREATE TABLE users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            age INTEGER
        )    
    """))

    # 데이터 삽입
    result = conn.execute(
        text("INSERT INTO users (name, age) VALUES (:name, :age)"),
        [
            {"name": "김정준", "age": 28},
            {"name": "김수림", "age": 27},
            {"name": "양정운", "age": 22},
        ]
    )

    print(f'🟦 {result.rowcount} 개의 행 INSERT')
    



2026-05-29 15:30:10,338 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-29 15:30:10,340 INFO sqlalchemy.engine.Engine 
        CREATE TABLE users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            age INTEGER
        )    
    
2026-05-29 15:30:10,340 INFO sqlalchemy.engine.Engine [generated in 0.00045s] ()
2026-05-29 15:30:10,343 INFO sqlalchemy.engine.Engine INSERT INTO users (name, age) VALUES (?, ?)
2026-05-29 15:30:10,343 INFO sqlalchemy.engine.Engine [generated in 0.00037s] [('김정준', 28), ('김수림', 27), ('양정운', 22)]
🟦 3 개의 행 INSERT
2026-05-29 15:30:10,343 INFO sqlalchemy.engine.Engine COMMIT


In [7]:
# 조회
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT name, age FROM users WHERE age > :min_age ORDER BY age"),
        {"min_age": 22},
    )

    for row in result:
        print(f"{row.name}: {row.age}")

2026-05-29 15:35:38,842 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-29 15:35:38,847 INFO sqlalchemy.engine.Engine SELECT name, age FROM users WHERE age > ? ORDER BY age
2026-05-29 15:35:38,848 INFO sqlalchemy.engine.Engine [generated in 0.00667s] (22,)
김수림: 27
김정준: 28
2026-05-29 15:35:38,852 INFO sqlalchemy.engine.Engine ROLLBACK


### 5.5 `text()`의 한계

`text()`는 간단하고 직접적이지만, 다음과 같은 한계가 있다.

- **데이터베이스 종속적**: 특정 DB의 SQL 방언이 그대로 노출됨
- **타입 안전성 없음**: 컬럼 이름 오타 등은 런타임에야 발견됨
- **재사용 어려움**: 동적으로 쿼리를 조립하기 까다로움

이 때문에 SQLAlchemy는 6장부터 다룰 **SQL Expression Language**를 권장한다. 다만 복잡한 raw SQL을 한 번 실행해야 할 때, 또는 마이그레이션이나 일회성 작업에는 `text()`가 여전히 유용하다.


---

## 6장. 메타데이터와 테이블 정의

### 6.1 메타데이터란

지금까지는 `text()`로 raw SQL을 실행했다. 이제 SQLAlchemy의 진짜 강점인 **SQL Expression Language**로 넘어간다. 그 출발점이 **메타데이터(Metadata)** 다.

메타데이터는 데이터베이스 스키마(테이블, 컬럼, 제약 조건 등)에 대한 Python 측 표현이다. 우리는 Python 코드로 테이블 구조를 선언하고, SQLAlchemy는 이 정보를 바탕으로 SQL을 생성한다.

### 6.2 `MetaData` 객체

`MetaData`는 여러 `Table` 객체를 모아두는 컨테이너다. 일반적으로 애플리케이션당 하나만 만든다.

```python
from sqlalchemy import MetaData

metadata = MetaData()
```

ORM을 사용할 때는 `DeclarativeBase`가 내부적으로 `MetaData`를 보유하므로 직접 만들 필요가 없다. 하지만 Core만 사용하거나 ORM의 동작을 이해하려면 이 객체를 먼저 알아야 한다.


### 6.3 `Table`정의하기

In [ ]:
from sqlalchemy import MetaData, Table, Column, Integer ,String, ForeignKey

metadata = MetaData()

user_table = Table(
    "users", # 물리적으로 생성될 테이를 이름
    metadata, # 소속 메타 데이터

    # 테이블의 컬럼들
    Column("id", Integer, primary_key=True), # PK
    Column("name", String(50), nullable=False), # NN 기본값은 nullable=True
    Column("email", String(120), unique=True), # UQ
    Column("age", Integer),
)

addresses_talbe = Table(
    "addresses",
    metadata,
    Column("id", Integer, primary_key=True),
    Column("user_id", ForeignKey("user.id", nullable=False),
    Column("city", String(50),
    Column("street", String(120),
)